In [2]:

import os, glob
from omegaconf import OmegaConf
# import pretty print
from pprint import pprint


In [3]:
# get all the result yaml files and parse them with omegaconf

def parse_results(file_path):
    with open(file_path, 'r') as f:
        conf = OmegaConf.load(f)
    return conf

def print_top_results(topdir):
    tasks = ['location','age','bmi','sex','supplement','helicobacter_pylori_infection','hbv_infection']
    tasks += ['hiv_infection', 'sunflower_seed_oil_emollient_therapy','chemotherapy','type_1_diabetes','crohns_disease', 'diarrhea']
    reg_tasks = ['age','bmi']
    class_tasks = [t for t in tasks if t not in reg_tasks]
    best_res_mae = {t:{} for t in reg_tasks}
    best_res_r2 = {t:{} for t in reg_tasks}
    best_res_weighted_auroc = {t:{} for t in class_tasks}
    best_res_accuracy = {t:{} for t in class_tasks}

    # get list of result files:
    for task in tasks:
        file_regex = f'{topdir}/{task}/*/best_model/test_metrics.yaml'
        res_files = glob.glob(file_regex)
        # get the top five results for each metric and store them in a dict
        
        for res_file in res_files:
            res = parse_results(res_file)
            # get the parent dir name which is the model name
            model_name = os.path.basename(os.path.dirname(os.path.dirname(res_file)))
            if task in reg_tasks:
                mae = res['test_mae']
                r2 = res['test_r2']
                best_res_mae[task][model_name] = mae
                best_res_r2[task][model_name] = r2
            else:
                best_res_weighted_auroc[task][model_name] = res['test_auroc_weighted'] if 'test_auroc_weighted' in res else res['test_auroc']
                best_res_accuracy[task][model_name] = res['test_accuracy']
        # keep the top five results for each metric
        print(80*'=')
        print(f'Top 5 results for {task}:')
        if task in reg_tasks:
            best_res_mae[task] = dict(sorted(best_res_mae[task].items(), key=lambda item: item[1])[:5])
            best_res_r2[task] = dict(sorted(best_res_r2[task].items(), key=lambda item: item[1], reverse=True)[:5])
            print(f'MAE:')
            pprint(sorted(best_res_mae[task].items(), key=lambda item: item[1]))
            # pprint(best_res_mae[task])
            print(f'R2:')
            # pprint(best_res_r2[task])
            pprint(sorted(best_res_r2[task].items(), key=lambda item: item[1], reverse=True))
        else:
            best_res_weighted_auroc[task] = dict(sorted(best_res_weighted_auroc[task].items(), key=lambda item: item[1], reverse=True)[:5])
            best_res_accuracy[task] = dict(sorted(best_res_accuracy[task].items(), key=lambda item: item[1], reverse=True)[:5])
            print('Weighted AUROC:')
            pprint(sorted(best_res_weighted_auroc[task].items(), key=lambda item: item[1], reverse=True))
            # pprint(best_res_weighted_auroc[task])
            print(f'Accuracy:')
            pprint(sorted(best_res_accuracy[task].items(), key=lambda item: item[1], reverse=True))
            # pprint(best_res_accuracy[task])



In [5]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/denoising/'
print_top_results(topdir)


Top 5 results for location:
Weighted AUROC:
[('log_rel_ab_7layers_multinomial_20_80_pool', 0.9328890411711711),
 ('clr_7layers_multinomial_20_80_cls', 0.9204068316798405),
 ('log_rel_ab_3layers_multinomial_20_80_cls', 0.9203347861566555),
 ('clr_7layers_multinomial_1_70_pool', 0.918769693716114),
 ('clr_7layers_multinomial_20_80_pool', 0.9166826406797612)]
Accuracy:
[('log_rel_ab_7layers_multinomial_20_80_pool', 0.8015653775322283),
 ('counts_7layers_multinomial_20_80_cls', 0.7955801104972375),
 ('counts_7layers_multinomial_20_80_pool', 0.7937384898710865),
 ('clr_7layers_multinomial_20_80_pool', 0.7886740331491713),
 ('clr_7layers_multinomial_1_70_pool', 0.7882136279926335)]
Top 5 results for age:
MAE:
[('counts_7layers_multinomial_20_80_cls', 18.94533348083496),
 ('log_rel_ab_7layers_multinomial_20_80_pool', 21.265344619750977),
 ('counts_3layers_multinomial_1_70_cls', 24.124935150146484),
 ('clr_7layers_multinomial_1_70_cls', 24.165876388549805),
 ('counts_3layers_multinomial_20_80_

In [16]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/gnn/'
print_top_results(topdir)


Top 5 results for location:
Weighted AUROC:
[('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.9443147343076785),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.9428848757079996),
 ('masking_log_rel_ab_len200_7layers_ratio_50_updated_cls', 0.9269682002885944),
 ('masking_log_rel_ab_len200_7layers_ratio_50_updated_pool', 0.926320362477706),
 ('masking_log_rel_ab_len200_7layers_ratio_70_updated_pool', 0.919813482210259)]
Accuracy:
[('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.8420810313075506),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.8218232044198895),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_pool',
  0.8080110497237569),
 ('masking_log_rel_ab_len200_7layers_ratio_70_updated_pool',
  0.8066298342541437),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.8052486187845304)]
Top 5 results for age:
MAE:
[('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 21.372346878051758),
 ('masking_clr_

In [17]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/evo/'
print_top_results(topdir)

Top 5 results for location:
Weighted AUROC:
[('masking_log_rel_ab_len200_7layers_ratio_50_evo_mlp_pool',
  0.9441047012144388),
 ('masking_clr_len200_7layers_ratio_70_evo_mlp_cls', 0.9354901969259205),
 ('masking_log_rel_ab_len200_3layers_ratio_50_evo_mlp_cls', 0.9302516160255747),
 ('masking_log_rel_ab_len200_3layers_ratio_70_evo_mlp_cls', 0.924240441292875),
 ('masking_clr_len200_3layers_ratio_50_evo_mlp_pool', 0.91945206897805)]
Accuracy:
[('masking_log_rel_ab_len200_3layers_ratio_50_evo_mlp_cls', 0.8176795580110497),
 ('masking_log_rel_ab_len200_3layers_ratio_70_evo_mlp_cls', 0.8139963167587477),
 ('masking_log_rel_ab_len200_7layers_ratio_50_evo_mlp_pool',
  0.8112338858195212),
 ('masking_log_rel_ab_len200_7layers_ratio_50_evo_mlp_cls', 0.8103130755064457),
 ('masking_log_rel_ab_len200_7layers_ratio_70_evo_mlp_cls', 0.7951197053406999)]


Top 5 results for age:
MAE:
[('masking_log_rel_ab_len200_7layers_ratio_70_evo_mlp_cls', 16.180870056152344),
 ('masking_clr_len200_7layers_ratio_70_evo_mlp_cls', 18.799348831176758),
 ('masking_clr_len200_3layers_ratio_70_evo_mlp_cls', 19.21967315673828),
 ('masking_log_rel_ab_len200_3layers_ratio_50_evo_mlp_cls', 20.103130340576172),
 ('masking_clr_len200_3layers_ratio_50_evo_mlp_cls', 21.282011032104492)]
R2:
[('masking_log_rel_ab_len200_7layers_ratio_70_evo_mlp_cls', -6.000815391540527),
 ('masking_clr_len200_7layers_ratio_70_evo_mlp_cls', -7.788727760314941),
 ('masking_clr_len200_3layers_ratio_70_evo_mlp_cls', -8.532157897949219),
 ('masking_log_rel_ab_len200_3layers_ratio_50_evo_mlp_cls', -8.981141090393066),
 ('masking_clr_len200_3layers_ratio_50_evo_mlp_cls', -10.130094528198242)]
Top 5 results for bmi:
MAE:
[('masking_clr_len200_7layers_ratio_50_evo_mlp_pool', 4.583160400390625),
 ('masking_log_rel_ab_len200_7layers_ratio_70_evo_mlp_pool', 4.77744722366333),
 ('masking_log_rel

In [6]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/final/'
print_top_results(topdir)

Top 5 results for location:
Weighted AUROC:
[('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.9480065946939529),
 ('masking_clr_len200_3layers_ratio_70_lr001_updated_cls', 0.9304291975800826),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_pool',
  0.9290952955011005),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.9264474083805936),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.9235336075183732)]
Accuracy:
[('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.8103130755064457),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.8029465930018416),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_pool',
  0.8001841620626151),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.7914364640883977),
 ('masking_clr_len200_3layers_ratio_50_lr001_updated_cls', 0.7863720073664825)]
Top 5 results for age:
MAE:
[('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  24.219650268554688),
 ('ma

In [4]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/test_random/'
print_top_results(topdir)


Top 5 results for location:
Weighted AUROC:
[('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.9393411277787775),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.9360131111666722),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_cls',
  0.9359904983718643),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.9351804660179488),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_pool',
  0.9233503663718274)]
Accuracy:
[('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.820902394106814),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.820902394106814),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_cls',
  0.8195211786372008),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.8167587476979742),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.7951197053406999)]
Top 5 results for age:
MAE:
[('masking_clr_len200_3layers_ratio_50_lr001_updated_cls', 19.12121582031

In [6]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/frozen_denoising/'
print_top_results(topdir)

Top 5 results for location:
Weighted AUROC:
[('counts_7layers_multinomial_20_80_pool', 0.7153015617726802),
 ('counts_7layers_multinomial_20_80_cls', 0.7098775657944492),
 ('counts_3layers_multinomial_1_70_pool', 0.702578059054478),
 ('counts_3layers_multinomial_20_80_pool', 0.702578059054478),
 ('clr_7layers_multinomial_1_70_pool', 0.6984145352349446)]
Accuracy:
[('counts_3layers_multinomial_1_70_pool', 0.6721915285451197),
 ('counts_3layers_multinomial_20_80_pool', 0.6721915285451197),
 ('counts_7layers_multinomial_20_80_cls', 0.6703499079189686),
 ('counts_3layers_multinomial_1_70_cls', 0.669889502762431),
 ('counts_3layers_multinomial_20_80_cls', 0.669889502762431)]
Top 5 results for age:
MAE:
[('counts_3layers_multinomial_1_70_pool', 15.719508171081543),
 ('counts_3layers_multinomial_20_80_pool', 15.719508171081543),
 ('clr_3layers_multinomial_1_70_pool', 26.53110122680664),
 ('clr_3layers_multinomial_20_80_pool', 26.53110122680664),
 ('counts_7layers_multinomial_20_80_pool', 28.7

In [7]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/test_random/'
print_top_results(topdir)

Top 5 results for location:
Weighted AUROC:
[('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.9393411277787775),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.9360131111666722),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_cls',
  0.9359904983718643),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.9351804660179488),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_pool',
  0.9233503663718274)]
Accuracy:
[('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.820902394106814),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.820902394106814),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_cls',
  0.8195211786372008),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.8167587476979742),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.7951197053406999)]
Top 5 results for age:
MAE:
[('masking_clr_len200_3layers_ratio_50_lr001_updated_cls', 19.12121582031

In [4]:
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/frozen_unmasking/'
print_top_results(topdir)

Top 5 results for location:
Weighted AUROC:
[('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.9393411277787775),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.9360131111666722),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_cls',
  0.9359904983718643),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.9351804660179488),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_pool',
  0.9233503663718274)]
Accuracy:
[('masking_log_rel_ab_len200_3layers_ratio_50_updated_cls', 0.820902394106814),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.820902394106814),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_cls',
  0.8195211786372008),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.8167587476979742),
 ('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.7951197053406999)]
Top 5 results for age:
MAE:
[('masking_clr_len200_3layers_ratio_50_lr001_updated_cls', 19.12121582031